In [ ]:
# --- repo root + config (walk parents; do not use ../..) ---
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# MedMentions generative inference (5 generative models)

Run generative models over the **existing** MedMentions validated perturbation set
(do **not** regenerate perturbations).

| Kind | Models |
|---|---|
| Causal instruct | BioMistral-7B, Mistral-7B-Instruct-v0.1, Llama3-OpenBioLLM-8B, Meta-Llama-3-8B-Instruct |
| Seq2seq | FLAN-T5-base (no chat template — CADEC seq2seq path) |

**Input:** archived `rq1_model_outputs.csv` / `rq1_validated_perturbations.csv`
(copied into `outputs/rq1/intermediate/`). Variant table uses `input_text` keyed by
`instance_id` + `input_variant_id` + `input_type`.

**Output:** `mm_model_raw/mm_raw_<key>.csv` → `rq1_generative_model_outputs.csv` →
reassembled `rq1_all_model_outputs.csv` (3 encoders + 5 generatives = 8 models).

In [1]:
# === Setup — absolute PROJECT_ROOT (nbconvert-safe); CUDA required ============
import json
import os
import shutil
import sys
import time
import gc
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer

PROJECT_ROOT = PROJECT_ROOT
CONFIG_PATH = PROJECT_ROOT / "config" / "config.json"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CONFIG_PATH:  {CONFIG_PATH}")
assert CONFIG_PATH.is_file(), f"Missing config.json: {CONFIG_PATH}"
assert torch.cuda.is_available(), "CUDA required — CPU placement is not allowed for this notebook"

with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)


def _expand_tree(obj):
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _model_src(key: str) -> str:
    if key not in CFG["models"]:
        raise KeyError(f"Unknown model key {key!r}. Choose from: {sorted(CFG['models'])}")
    return CFG["models"][key]


def assert_model_on_cuda(model, name: str):
    """Crash loudly if the model landed on CPU."""
    devices = {p.device for p in model.parameters()}
    cuda_devs = {d for d in devices if d.type == "cuda"}
    assert cuda_devs, (
        f"CPU PLACEMENT BUG: {name} has no parameters on CUDA. "
        f"Devices seen: {sorted(str(d) for d in devices)}. "
        f"Refuse to run silently on CPU."
    )
    first = next(model.parameters()).device
    if len(devices) == 1:
        assert first.type == "cuda", (
            f"CPU PLACEMENT BUG: {name} first parameter on {first}, expected cuda"
        )
    print(f"DEVICE OK [{name}]: param_devices={sorted(str(d) for d in devices)} | GPU={torch.cuda.get_device_name(0)}")


def load_generative(key: str):
    """Causal 7–8B — bf16 + device_map=auto (same as CADEC_inference)."""
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    assert_model_on_cuda(model, f"causal:{key}")
    return tokenizer, model


def load_seq2seq(key: str):
    """FLAN-T5 — NO device_map; explicit .to('cuda') (same as CADEC_inference)."""
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
    )
    model = model.to("cuda").eval()
    assert_model_on_cuda(model, f"seq2seq:{key}")
    return tokenizer, model


def free_model(model):
    del model
    torch.cuda.empty_cache()


print(f"GPU: {torch.cuda.get_device_name(0)}")
sys.stdout.flush()


/home/s224858267/.conda/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs
CONFIG_PATH:  /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/config.json
GPU: NVIDIA L40S


## 1) Copy archive inputs + build MedMentions variant table

Uses the existing encoder `rq1_model_outputs.csv` so `input_text` / variant IDs match
the MedMentions encoder run exactly. Does **not** regenerate perturbations.

In [2]:
# === Paths, copy archive inputs, build variant table ========================
def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()


INTER_DIR = PROJECT_ROOT / "outputs" / "rq1" / "intermediate"
INTER_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = INTER_DIR / "mm_model_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = INTER_DIR / "rq1_generative_model_outputs.csv"

ARCHIVE_INTER = (
    PROJECT_ROOT
    / "outputs_archive"
    / "outputs_pre_rerun_20260731"
    / "rq1"
    / "intermediate"
)
SRC_VALIDATED = ARCHIVE_INTER / "rq1_validated_perturbations.csv"
SRC_ENCODER_OUT = ARCHIVE_INTER / "rq1_model_outputs.csv"
DST_VALIDATED = INTER_DIR / "rq1_validated_perturbations.csv"
DST_ENCODER_OUT = INTER_DIR / "rq1_model_outputs.csv"

for src, dst, label in [
    (SRC_VALIDATED, DST_VALIDATED, "validated perturbations"),
    (SRC_ENCODER_OUT, DST_ENCODER_OUT, "encoder model outputs"),
]:
    assert src.is_file(), f"Missing archive {label}: {src}"
    if not dst.is_file() or dst.stat().st_size == 0:
        shutil.copy2(src, dst)
        _log(f"Copied {label}: {src.name} → {dst}")
    else:
        _log(f"Keep existing {dst.name} ({dst.stat().st_size:,} bytes)")

CAUSAL_MODELS = [
    {"key": "biomistral", "model_name": "BioMistral-7B", "domain": "biomedical"},
    {"key": "mistral", "model_name": "Mistral-7B-Instruct-v0.1", "domain": "general"},
    {"key": "openbiollm", "model_name": "Llama3-OpenBioLLM-8B", "domain": "biomedical"},
    {"key": "llama3", "model_name": "Meta-Llama-3-8B-Instruct", "domain": "general"},
]
SEQ2SEQ_MODELS = [
    {"key": "flan-t5-base", "model_name": "FLAN-T5-base", "domain": "general"},
]
ALL_GENERATIVE = CAUSAL_MODELS + SEQ2SEQ_MODELS
MATCHED_PAIRS = [
    ("BioMistral-7B", "Mistral-7B-Instruct-v0.1"),
    ("Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct"),
]

print("Models to run (generative only):")
for s in CAUSAL_MODELS:
    print(f"  causal  {s['model_name']:28s} key={s['key']} src={_model_src(s['key'])}")
for s in SEQ2SEQ_MODELS:
    print(f"  seq2seq {s['model_name']:28s} key={s['key']} src={_model_src(s['key'])}")

# Variant table from encoder outputs — one row per (instance, variant), with input_text
df_enc = pd.read_csv(DST_ENCODER_OUT)
need = [
    "instance_id", "input_variant_id", "input_type", "input_text",
    "gold_cui_or_entity", "perturbation_type",
]
missing = [c for c in need if c not in df_enc.columns]
assert not missing, f"Encoder outputs missing columns: {missing}"

df_variants = (
    df_enc[need]
    .drop_duplicates(subset=["instance_id", "input_variant_id", "input_type"])
    .reset_index(drop=True)
)
# Prefer m>=3 instances (matches entropy inclusion); keep originals of those instances
_m = df_variants.groupby("instance_id").size()
keep_ids = set(_m[_m >= 3].index)
n_before = len(df_variants)
df_variants = df_variants[df_variants["instance_id"].isin(keep_ids)].reset_index(drop=True)
_log(
    f"Variant table: {len(df_variants):,} rows (from {n_before:,}) | "
    f"instances={df_variants['instance_id'].nunique():,} | "
    f"m>=3 kept={len(keep_ids):,} excluded={(_m < 3).sum()}"
)
print(df_variants["input_type"].value_counts().to_string())
assert len(df_variants) > 0
sys.stdout.flush()


[2026-08-02 00:17:48 UTC] Keep existing rq1_validated_perturbations.csv (12,253,122 bytes)


[2026-08-02 00:17:48 UTC] Keep existing rq1_model_outputs.csv (22,781,122 bytes)


Models to run (generative only):
  causal  BioMistral-7B                key=biomistral src=/home/s224858267/data/models/BioMistral-7B
  causal  Mistral-7B-Instruct-v0.1     key=mistral src=/home/s224858267/data/models/Mistral-7B-Instruct-v0.1
  causal  Llama3-OpenBioLLM-8B         key=openbiollm src=/home/s224858267/data/models/Llama3-OpenBioLLM-8B
  causal  Meta-Llama-3-8B-Instruct     key=llama3 src=/home/s224858267/data/models/Meta-Llama-3-8B-Instruct
  seq2seq FLAN-T5-base                 key=flan-t5-base src=google/flan-t5-base
[2026-08-02 00:17:48 UTC] Variant table: 3,208 rows (from 3,211) | instances=547 | m>=3 kept=547 excluded=3


input_type
perturbation    2661
original         547


## 2) Inference helpers (causal chat-template + FLAN seq2seq from CADEC_inference)

In [3]:
# === Causal generative inference (CADEC_inference generate_concept) ==========
CAUSAL_MAX_NEW_TOKENS = 16  # short concept span, not prose
_CONCEPT_INSTR = (
    "Identify the primary medical concept in the following clinical text. "
    "Reply with only the concept name.\n\n"
    "Text: {text}"
)


def raw_path_for(key: str) -> Path:
    return RAW_DIR / f"mm_raw_{key}.csv"


def maybe_skip_existing(spec: dict) -> Path | None:
    """Resume: if mm_raw_<key>.csv exists and is non-empty, skip inference."""
    out_path = raw_path_for(spec["key"])
    if out_path.exists() and out_path.stat().st_size > 0:
        n_rows = sum(1 for _ in open(out_path, encoding="utf-8", errors="replace")) - 1
        _log(
            f"SKIP {spec['model_name']} — exists {out_path.name} "
            f"({out_path.stat().st_size:,} bytes, ~{n_rows:,} rows)"
        )
        return out_path
    return None


# Llama-3 chat template (OpenBioLLM ships without one)
_LLAMA3_CHAT_TEMPLATE = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    "+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}"
    "{{ content }}{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)


def _ensure_chat_template(tokenizer) -> None:
    """Attach a chat template when missing (OpenBioLLM = Llama-3 family)."""
    if getattr(tokenizer, "chat_template", None):
        return
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    unk = getattr(tokenizer, "unk_token_id", None)
    if eot is not None and eot != unk:
        tokenizer.chat_template = _LLAMA3_CHAT_TEMPLATE
        return
    tokenizer.chat_template = (
        "{{ bos_token }}{% for message in messages %}"
        "{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}"
        "{% elif message['role'] == 'assistant' %}{{ message['content'] }}"
        "{% endif %}{% endfor %}"
    )


def _clean_concept_output(decoded: str) -> str:
    """Strip residual chat-template / scaffolding markers from decoded span."""
    text = decoded.strip()
    for marker in ("[/INST]", "</s>", "<s>"):
        text = text.replace(marker, " ")
    for marker in ("Answer:", "Concept:", "The primary medical concept is"):
        if marker in text:
            text = text.split(marker)[-1]
    m = re.search(
        r"(?is)the primary medical concepts?\b.*?\b(?:is|are)\b\s*:?\s*",
        text,
    )
    if m:
        text = text[m.end():]
    text = text.strip(" \"'`.")
    text = " ".join(text.split()).strip()
    return text[:200]


def generate_concept(text: str, tokenizer, model) -> str:
    _ensure_chat_template(tokenizer)
    user_content = _CONCEPT_INSTR.format(text=text)
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512, padding=True)
    try:
        first_dev = next(model.parameters()).device
        enc = {k: v.to(first_dev) for k, v in enc.items()}
    except StopIteration:
        enc = {k: v.to("cuda:0") for k, v in enc.items()}

    input_len = enc["input_ids"].shape[-1]
    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    unk = getattr(tokenizer, "unk_token_id", None)
    if eot is not None and eot != unk and eot not in eos_ids:
        eos_ids.append(eot)
    gen_kwargs = dict(
        max_new_tokens=CAUSAL_MAX_NEW_TOKENS,
        do_sample=False,  # greedy, T=0
        pad_token_id=tokenizer.pad_token_id,
    )
    if eos_ids:
        gen_kwargs["eos_token_id"] = eos_ids if len(eos_ids) > 1 else eos_ids[0]
    with torch.no_grad():
        out = model.generate(**enc, **gen_kwargs)
    new_tokens = out[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return _clean_concept_output(decoded)


def _rows_from_generations(spec, gens):
    rows = []
    for (_, row), gen in zip(df_variants.iterrows(), gens):
        rows.append({
            "instance_id": row["instance_id"],
            "model_name": spec["model_name"],
            "input_variant_id": row["input_variant_id"],
            "input_type": row["input_type"],
            "output_text": gen,
            "gold_cui_or_entity": row["gold_cui_or_entity"],
            "perturbation_type": row["perturbation_type"],
        })
    return rows


def run_one_causal(spec: dict) -> Path:
    key, model_name = spec["key"], spec["model_name"]
    skipped = maybe_skip_existing(spec)
    if skipped is not None:
        return skipped
    out_path = raw_path_for(key)

    src = _model_src(key)
    _log(f"LOAD causal {model_name} from {src} (bf16, device_map=auto, T=0 greedy)")
    t0 = time.perf_counter()
    tokenizer, model = load_generative(key)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    _ensure_chat_template(tokenizer)

    gens = []
    for i, row in tqdm(df_variants.iterrows(), total=len(df_variants), desc=model_name):
        text = str(row["input_text"]) if pd.notna(row["input_text"]) else ""
        try:
            gens.append(generate_concept(text, tokenizer, model))
        except Exception as e:
            _log(f"WARN generate failed {model_name} row={i}: {e}")
            gens.append("")
        if (len(gens) % 500) == 0:
            _log(f"  {model_name}: {len(gens)}/{len(df_variants)} elapsed={time.perf_counter()-t0:.0f}s")

    df_out = pd.DataFrame(_rows_from_generations(spec, gens))
    df_out.to_csv(out_path, index=False)
    n_unique = df_out["output_text"].fillna("").astype(str).nunique()
    _log(
        f"DONE {model_name}: rows={len(df_out)} unique_outputs={n_unique} "
        f"mean_len={df_out['output_text'].astype(str).str.len().mean():.1f} -> {out_path.name}"
    )
    if n_unique < 200:
        _log(f"WARN degeneracy check: {model_name} unique_outputs={n_unique} (want thousands, not ~140)")

    free_model(model)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


# FLAN-T5 seq2seq (CADEC_inference — no chat template)
MAX_NEW_TOKENS = 32  # same as CADEC FLAN path


def generate_concept_seq2seq_batch(texts, tokenizer, model, batch_size=32):
    """Batched FLAN-T5 greedy decode on CUDA."""
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) if t is not None else "" for t in texts[i:i + batch_size]]
        prompts = [
            "Identify the primary medical concept in the following clinical text. "
            "Reply with only the concept name.\n\n"
            f"Text: {t}"
            for t in batch
        ]
        enc = tokenizer(
            prompts, return_tensors="pt", truncation=True, max_length=512, padding=True
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        outs.extend([d.strip()[:200] for d in decoded])
    return outs


def run_one_seq2seq(spec: dict) -> Path:
    key, model_name = spec["key"], spec["model_name"]
    skipped = maybe_skip_existing(spec)
    if skipped is not None:
        return skipped
    out_path = raw_path_for(key)

    src = _model_src(key)
    _log(f"LOAD seq2seq {model_name} from {src} (bf16, explicit cuda, T=0 greedy)")
    t0 = time.perf_counter()
    tokenizer, model = load_seq2seq(key)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    texts = df_variants["input_text"].fillna("").astype(str).tolist()
    _log(f"  Batched seq2seq generate on cuda, n={len(texts):,} batch_size=32")
    gens = generate_concept_seq2seq_batch(texts, tokenizer, model, batch_size=32)

    df_out = pd.DataFrame(_rows_from_generations(spec, gens))
    df_out.to_csv(out_path, index=False)
    n_unique = df_out["output_text"].fillna("").astype(str).nunique()
    _log(
        f"DONE {model_name}: rows={len(df_out)} unique_outputs={n_unique} "
        f"mean_len={df_out['output_text'].astype(str).str.len().mean():.1f} -> {out_path.name}"
    )
    if n_unique < 200:
        _log(f"WARN degeneracy check: {model_name} unique_outputs={n_unique}")

    free_model(model)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


## 3) Run generative models (causal + FLAN seq2seq; resume via `maybe_skip_existing`)

In [4]:
# === Run generative models sequentially =====================================
for spec in CAUSAL_MODELS:
    run_one_causal(spec)
for spec in SEQ2SEQ_MODELS:
    run_one_seq2seq(spec)
_log("All generative models processed — existing mm_raw_*.csv files were skipped.")


[2026-08-02 00:17:48 UTC] SKIP BioMistral-7B — exists mm_raw_biomistral.csv (307,902 bytes, ~3,208 rows)


[2026-08-02 00:17:48 UTC] SKIP Mistral-7B-Instruct-v0.1 — exists mm_raw_mistral.csv (358,279 bytes, ~3,208 rows)


[2026-08-02 00:17:48 UTC] SKIP Llama3-OpenBioLLM-8B — exists mm_raw_openbiollm.csv (352,350 bytes, ~3,208 rows)


[2026-08-02 00:17:48 UTC] SKIP Meta-Llama-3-8B-Instruct — exists mm_raw_llama3.csv (362,920 bytes, ~3,208 rows)


[2026-08-02 00:17:48 UTC] LOAD seq2seq FLAN-T5-base from google/flan-t5-base (bf16, explicit cuda, T=0 greedy)


...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


DEVICE OK [seq2seq:flan-t5-base]: param_devices=['cuda:0'] | GPU=NVIDIA L40S
[2026-08-02 00:17:55 UTC]   Batched seq2seq generate on cuda, n=3,208 batch_size=32


[2026-08-02 00:18:09 UTC] DONE FLAN-T5-base: rows=3208 unique_outputs=1024 mean_len=10.9 -> mm_raw_flan-t5-base.csv


[2026-08-02 00:18:09 UTC] Freed FLAN-T5-base


[2026-08-02 00:18:09 UTC] All generative models processed — existing mm_raw_*.csv files were skipped.


## 4) Assemble generative CSV + `rq1_all_model_outputs.csv` (8 models)

In [5]:
# === Concatenate generatives + reassemble all 8 model outputs ===============
TARGET_COLS = [
    "instance_id",
    "model_name",
    "input_variant_id",
    "input_type",
    "output_text",
    "gold_cui_or_entity",
    "perturbation_type",
]

parts = []
print("===== Per-model unique-output degeneracy check =====")
for spec in ALL_GENERATIVE:
    p = raw_path_for(spec["key"])
    assert p.is_file() and p.stat().st_size > 0, f"Missing model output: {p}"
    df_p = pd.read_csv(p)
    missing = [c for c in TARGET_COLS if c not in df_p.columns]
    assert not missing, f"{p.name} missing cols {missing}"
    n_unique = df_p["output_text"].fillna("").astype(str).nunique()
    empty_frac = (df_p["output_text"].fillna("").astype(str).str.strip() == "").mean()
    mean_len = df_p["output_text"].fillna("").astype(str).str.len().mean()
    print(
        f"{spec['model_name']:28s}  rows={len(df_p):5d}  unique={n_unique:5d}  "
        f"empty_frac={empty_frac:.3f}  mean_len={mean_len:.1f}"
    )
    assert n_unique >= 200, (
        f"DEGENERACY: {spec['model_name']} unique_outputs={n_unique} (want thousands, not ~140)"
    )
    parts.append(df_p[TARGET_COLS])
    _log(f"Loaded {p.name}: {len(df_p):,} rows")

df_gen = pd.concat(parts, ignore_index=True)
print(f"\nWriting: {OUT_CSV}")
df_gen.to_csv(OUT_CSV, index=False)
_log(f"Wrote {len(df_gen):,} rows → {OUT_CSV}")
print(df_gen.groupby("model_name").size().to_string())
sys.stdout.flush()

print("\n===== Matched-pair output divergence asserts =====")
for bio_name, gen_name in MATCHED_PAIRS:
    a = df_gen[df_gen["model_name"] == bio_name].sort_values(["instance_id", "input_variant_id"])
    b = df_gen[df_gen["model_name"] == gen_name].sort_values(["instance_id", "input_variant_id"])
    assert len(a) > 0 and len(b) > 0, f"Missing outputs for pair {bio_name} vs {gen_name}"
    merged = a.merge(
        b,
        on=["instance_id", "input_variant_id", "input_type"],
        suffixes=("_bio", "_gen"),
        how="inner",
    )
    identical_frac = (
        merged["output_text_bio"].fillna("").astype(str)
        == merged["output_text_gen"].fillna("").astype(str)
    ).mean()
    print(f"{bio_name} vs {gen_name}: identical_frac={identical_frac:.4f} n={len(merged):,}")
    assert identical_frac < 1.0, (
        f"IDENTICAL-OUTPUT BUG: {bio_name} vs {gen_name} produced 100% identical output_text"
    )

# --- Reassemble rq1_all_model_outputs.csv (3 encoders + 5 generatives) --------
ALL_OUT = INTER_DIR / "rq1_all_model_outputs.csv"
df_enc_full = pd.read_csv(DST_ENCODER_OUT)
enc_parts = []
for model_name, g in df_enc_full.groupby("model_name"):
    enc_parts.append(pd.DataFrame({
        "instance_id": g["instance_id"].values,
        "model_name": model_name,
        "input_variant_id": g["input_variant_id"].values,
        "input_type": g["input_type"].values,
        "perturbation_type": g["perturbation_type"].values,
        "gold_cui_or_entity": g["gold_cui_or_entity"].values,
        # Encoder "output" is already a CUI (PART2 mixed mapper uses this directly)
        "output_text": g["predicted_cui_or_cluster"].astype(str).values,
    }))
df_enc_std = pd.concat(enc_parts, ignore_index=True)

# Align generative to m>=3 variant keys used in inference; encoders may have m<3 rows —
# keep encoder rows as-is (PART2 applies m>=3 at entropy time), generative already filtered.
df_all8 = pd.concat([df_enc_std[TARGET_COLS], df_gen[TARGET_COLS]], ignore_index=True)
df_all8.to_csv(ALL_OUT, index=False)
_log(f"Wrote {len(df_all8):,} rows → {ALL_OUT}")
print("\n===== rq1_all_model_outputs.csv model counts =====")
print(df_all8.groupby("model_name").size().to_string())
n_models = df_all8["model_name"].nunique()
assert n_models == 8, f"Expected 8 models, got {n_models}: {sorted(df_all8['model_name'].unique())}"
_log("ASSERT OK: 8-model MedMentions outputs assembled (3 encoders + 5 generatives).")
sys.stdout.flush()


===== Per-model unique-output degeneracy check =====


BioMistral-7B                 rows= 3208  unique= 1026  empty_frac=0.002  mean_len=10.4
[2026-08-02 00:18:09 UTC] Loaded mm_raw_biomistral.csv: 3,208 rows


Mistral-7B-Instruct-v0.1      rows= 3208  unique= 1039  empty_frac=0.000  mean_len=15.1
[2026-08-02 00:18:09 UTC] Loaded mm_raw_mistral.csv: 3,208 rows


Llama3-OpenBioLLM-8B          rows= 3208  unique= 1367  empty_frac=0.000  mean_len=17.3
[2026-08-02 00:18:09 UTC] Loaded mm_raw_openbiollm.csv: 3,208 rows


Meta-Llama-3-8B-Instruct      rows= 3208  unique= 1010  empty_frac=0.002  mean_len=16.5
[2026-08-02 00:18:09 UTC] Loaded mm_raw_llama3.csv: 3,208 rows


...

model_name
BioMistral-7B               3208
FLAN-T5-base                3208
Llama3-OpenBioLLM-8B        3208
Meta-Llama-3-8B-Instruct    3208
Mistral-7B-Instruct-v0.1    3208



===== Matched-pair output divergence asserts =====
BioMistral-7B vs Mistral-7B-Instruct-v0.1: identical_frac=0.2640 n=3,208
Llama3-OpenBioLLM-8B vs Meta-Llama-3-8B-Instruct: identical_frac=0.1306 n=3,208


[2026-08-02 00:18:09 UTC] Wrote 25,673 rows → /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/intermediate/rq1_all_model_outputs.csv



===== rq1_all_model_outputs.csv model counts =====


model_name
BERT-base                   3211
BioBERT                     3211
BioMistral-7B               3208
FLAN-T5-base                3208
Llama3-OpenBioLLM-8B        3208
Meta-Llama-3-8B-Instruct    3208
Mistral-7B-Instruct-v0.1    3208
PubMedBERT                  3211
[2026-08-02 00:18:09 UTC] ASSERT OK: 8-model MedMentions outputs assembled (3 encoders + 5 generatives).
